In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM, GPT2Tokenizer

In [ ]:
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

plt.rcParams.update({
    'figure.facecolor': '#282a2c',
    'figure.edgecolor': '#282a2c',
    'axes.facecolor':   '#282a2c',
    'axes.edgecolor':   '#DDE2F4',
    'axes.labelcolor':  '#DDE2F4',
    'xtick.color':      '#DDE2F4',
    'ytick.color':      '#DDE2F4',
    'text.color':       '#DDE2F4',
    'axes.spines.right': False,
    'axes.spines.top':   False,
    'axes.titleweight': 'bold',
    'axes.labelweight': 'bold',
    'savefig.dpi':300,
})

In [ ]:
# softmax
z = [1,2,3,4,5,6,7,8,9,10]

num = np.exp(z)
den = np.sum( np.exp(z) )
sm = num / den

print(sm)
print(np.sum(sm))

In [ ]:
zTorch = torch.tensor(z,dtype=torch.float32)

zTorch_sm = F.softmax(zTorch,dim=-1)
zTorch_sm

In [ ]:
plt.figure(figsize=(10,4))

plt.plot(z,sm,'ks-',markerfacecolor=[.9,.7,.7],markersize=10,label='Manual')
plt.plot(z,zTorch_sm,'bx:',markersize=8,label='PyTorch')
plt.legend()

plt.gca().set(xlabel='Original number (z)',ylabel='Softmax probability $\\sigma (z)$',
              title='$\\sum\\sigma (z)$ = %g' %np.sum(sm))

plt.tight_layout()
plt.savefig('softmax.png')
plt.show()

In [ ]:
x = torch.linspace(-5,5,55)

shapes = 'soh^'

plt.figure(figsize=(10,4))
for i,temp in enumerate([.3,.6,1,1.4]):
  sm = F.softmax(x/temp,dim=-1)
  plt.plot(x,sm,shapes[i]+'-',linewidth=2,label='T = %g' %temp)

plt.legend()
plt.gca().set(xlabel='Original number',ylabel='Softmax probability')
# plt.yscale('log') # FYI

plt.tight_layout()
plt.savefig('temp.png')
plt.show()

In [ ]:
gpt2_small = AutoModelForCausalLM.from_pretrained('gpt2')
gpt2_large = AutoModelForCausalLM.from_pretrained('gpt2-large')

gpt2_small.eval()
gpt2_large.eval()

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

In [ ]:
print(f'GPT-2-small has {gpt2_small.num_parameters():,} parameters.')
print(f'GPT-2-large has {gpt2_large.num_parameters():,} parameters.')

In [ ]:
txt = 'The band saw created a lot of heat, perhaps even melting the rock.'
tokens = tokenizer.encode(txt,return_tensors='pt') # pt = PyTorch
tokens

In [ ]:
print(f'The text comprises {tokens.shape[1]} tokens.\n')

for t in tokens[0]:
  print(f'{t:5} is "{tokenizer.decode(t)}"')

In [ ]:
# forward pass through the model
outputs = gpt2_small(tokens)
outputs

In [ ]:
outputs.logits.shape

In [ ]:
logits = outputs.logits[0,-1,:].detach()
logits_sm = F.softmax(logits,dim=-1)
logits.shape

In [ ]:
print(f'The sum of the raw logits is {logits.sum():.3f}')
print(f'The sum of the softmax logits is {logits_sm.sum():.3f}')

In [ ]:
_,axs = plt.subplots(1,3,figsize=(12,3))

axs[0].plot(logits,'ks',markerfacecolor=[.9,.7,.7,.3])
axs[0].set(xlim=[-10,tokenizer.vocab_size+9],xlabel='Token index',
           ylabel='Output logits',title='A) All final token logits')

axs[1].plot(logits_sm,'o',markerfacecolor=[.7,.9,.7,.3])
axs[1].set(xlim=[-10,tokenizer.vocab_size+9],xlabel='Token index',
           ylabel='Probabilities',title='B) Softmax probabilities')

axs[2].plot(logits,logits_sm,'^',markerfacecolor=[.7,.7,.9,.7])
axs[2].set(xlabel='"Raw" logits',ylabel='Softmax logits',
           title='C) Logits by probabilities')

plt.tight_layout()
plt.savefig('softmax_logits.png')
plt.show()

In [ ]:
max_logit = logits_sm.argmax()
print(f'The maximum softmax logit is #{max_logit} with a value of {logits_sm[max_logit]:.3f}')
print(f'The max word is "{tokenizer.decode(max_logit)}"')

In [ ]:
k = 11
top_k = torch.topk(logits_sm,k)

print(txt,'___\n')

for i in range(k):
  val = top_k.values[i]
  tok = top_k.indices[i]
  print(f'{tok:5} ({100*val:4.1f}%) is "{tokenizer.decode(tok)}"')

In [ ]:
temps = [ .5,1,1.5 ]

plt.figure(figsize=(10,4))

shapes = 'so^'

for i,T in enumerate(temps):
  sm = F.softmax(logits/T,dim=-1)
  top_k = torch.topk(sm,k)

  color = [.7,.7,.7]
  color[i] = .9
  plt.plot(top_k.values,f'{shapes[i]}-',markerfacecolor=color,
           color=color,markeredgecolor='k',markersize=10,label=f'T = {T}')


plt.legend()
plt.gca().set(xlabel=f'Top-{k} indices',ylabel='Softmax probabilities (log)',yscale='log')

plt.tight_layout()
plt.savefig('top7_temp.png')
plt.show()

In [ ]:
tokens = tokenizer.encode('A plethora of platypuses.',return_tensors='pt')
outputs_small = gpt2_small(tokens)
outputs_large = gpt2_large(tokens)

In [ ]:
logits_small = outputs_small.logits[0,-1,:].detach()
logits_large = outputs_large.logits[0,-1,:].detach()

_,axs = plt.subplots(1,3,figsize=(12,3.5))

axs[0].plot(logits_small,'k.',alpha=.2)
axs[0].set(xlim=[-10,tokenizer.vocab_size+9],xlabel='Token index',ylabel='Output logits',title='A) GPT-2 SMALL')

axs[1].plot(logits_large,'k.',alpha=.2)
axs[1].set(xlim=[-10,tokenizer.vocab_size+9],xlabel='Token index',ylabel='Output logits',title='B) GPT-2 LARGE')

axs[2].plot(logits_small,logits_large,'m.',alpha=.2)
axs[2].set(xlabel='GPT-2 SMALL',ylabel='GPT-2 LARGE',title='C) Comparison of both models')

plt.tight_layout()
plt.savefig('gpt2_s_l.png')
plt.show()

In [ ]:
sm_manual_small = torch.exp(logits_small) / torch.sum(torch.exp(logits_small))
sm_manual_large = torch.exp(logits_large) / torch.sum(torch.exp(logits_large))

_,axs = plt.subplots(1,3,figsize=(12,3.5))

axs[0].plot(sm_manual_small,'k.',alpha=.2)
axs[0].set(xlim=[-10,tokenizer.vocab_size+9],xlabel='Token index',ylabel='Softmax probabilities',title='A) GPT-2 SMALL')

axs[1].plot(sm_manual_large,'k.',alpha=.2)
axs[1].set(xlim=[-10,tokenizer.vocab_size+9],xlabel='Token index',ylabel='Softmax probabilities',title='B) GPT-2 LARGE')

axs[2].plot(sm_manual_small,sm_manual_large,'m.',alpha=.2)
axs[2].set(xlabel='GPT-2 SMALL',ylabel='GPT-2 LARGE',title='C) Comparison of both models')

plt.tight_layout()
plt.savefig('gpt2_s_l_manual.png')
plt.show()

In [ ]:
logits_small[3000],sm_manual_small[1000]

In [ ]:
# simple normalization (subtract max value)
logits_small_norm = logits_small - logits_small.max()
logits_large_norm = logits_large - logits_large.max()

_,axs = plt.subplots(1,3,figsize=(12,3.5))

axs[0].plot(logits_small_norm,'k.',alpha=.2)
axs[0].set(xlim=[-10,tokenizer.vocab_size+9],xlabel='Token index',ylabel='Raw logits (max-norm)',title='A) GPT-2 SMALL')

axs[1].plot(logits_large_norm,'k.',alpha=.2)
axs[1].set(xlim=[-10,tokenizer.vocab_size+9],xlabel='Token index',ylabel='Raw logits (max-norm)',title='B) GPT-2 LARGE')

axs[2].plot(logits_small_norm,logits_large_norm,'m.',alpha=.2)
axs[2].set(xlabel='GPT-2 SMALL',ylabel='GPT-2 LARGE',title='C) Comparison of both models')

plt.tight_layout()
plt.savefig('norm_gpt2_s_l.png')
plt.show()

In [ ]:
sm_manual_smallN = torch.exp(logits_small_norm) / torch.sum(torch.exp(logits_small_norm))
sm_manual_largeN = torch.exp(logits_large_norm) / torch.sum(torch.exp(logits_large_norm))

_,axs = plt.subplots(1,3,figsize=(12,3.5))

axs[0].plot(sm_manual_smallN,'k.',alpha=.2)
axs[0].set(xlim=[-10,tokenizer.vocab_size+9],xlabel='Token index',ylabel='Softmax probabilities',title='A) GPT-2 SMALL')

axs[1].plot(sm_manual_largeN,'k.',alpha=.2)
axs[1].set(xlim=[-10,tokenizer.vocab_size+9],xlabel='Token index',ylabel='Softmax probabilities',title='B) GPT-2 LARGE')

axs[2].plot(sm_manual_smallN,sm_manual_largeN,'m.',alpha=.2)
axs[2].set(xlabel='GPT-2 SMALL',ylabel='GPT-2 LARGE',title='C) Comparison of both models')

plt.tight_layout()
plt.savefig('norm_gpt2_s_l_manual.png')
plt.show()

In [ ]:
sm_torch_small = F.softmax(logits_small,dim=-1)
sm_torch_large = F.softmax(logits_large,dim=-1)

_,axs = plt.subplots(1,3,figsize=(12,3.5))

axs[0].plot(sm_torch_small,'k.',alpha=.2)
axs[0].set(xlim=[-10,tokenizer.vocab_size+9],xlabel='Token index',ylabel='Softmax probabilities',title='GPT-2 SMALL')

axs[1].plot(sm_torch_large,'k.',alpha=.2)
axs[1].set(xlim=[-10,tokenizer.vocab_size+9],xlabel='Token index',ylabel='Softmax probabilities',title='GPT-2 LARGE')

axs[2].plot(sm_torch_small,sm_torch_large,'m.',alpha=.2)
axs[2].set(xlabel='GPT-2 SMALL',ylabel='GPT-2 LARGE',title='Comparison of both models')

plt.tight_layout()
plt.savefig('pytorch_gpt2_s_l.png')
plt.show()